<a href="https://colab.research.google.com/github/teju917/GENAITUTORIALS050/blob/main/RAG_Build_Evaluate_Hallucination_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG System: Build, Evaluate, and Hallucination Check

This notebook uses four main code cells:
1. Build a RAG system from uploaded PDF/TXT documents.
2. Evaluate baseline answers versus RAG answers and retrieval quality.
3. Check answer faithfulness / unsupported claims using an NLI model.
4. Display results and calculate hallucination rates.

**Run cells from top to bottom.** Upload at least one PDF or TXT file when prompted. The evaluation questions are generated from your document chunks; edit them for your own experiment if needed.

In [ ]:
# CELL 1 — BUILD RAG (install, load documents, chunk, embed, retrieve, generate)

!pip -q install -U transformers sentence-transformers pypdf accelerate pandas scikit-learn

import os, re, glob, textwrap, warnings
import pandas as pd
import torch
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Upload PDF and/or TXT files
uploaded = files.upload()
documents = []
for filename in uploaded:
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)
        content = "\n".join(page.extract_text() or "" for page in reader.pages)
    elif filename.lower().endswith(".txt"):
        with open(filename, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
    else:
        print(f"Skipping unsupported file: {filename}")
        continue
    content = re.sub(r"\s+", " ", content).strip()
    if content:
        documents.append({"source": filename, "text": content})

if not documents:
    raise ValueError("No readable PDF/TXT content found. Upload a text-based PDF or TXT file.")

# Split documents into overlapping word chunks
CHUNK_SIZE, OVERLAP = 180, 35
chunks = []
for doc in documents:
    words = doc["text"].split()
    start = 0
    while start < len(words):
        part = words[start:start + CHUNK_SIZE]
        if not part:
            break
        chunks.append({
            "id": len(chunks),
            "source": doc["source"],
            "text": " ".join(part)
        })
        if start + CHUNK_SIZE >= len(words):
            break
        start += CHUNK_SIZE - OVERLAP

if not chunks:
    raise ValueError("Could not create chunks from uploaded documents.")

# Embedding model and vector index
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
)

def retrieve(query, top_k=3):
    q_emb = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = cosine_similarity(q_emb, chunk_embeddings)[0]
    indices = scores.argsort()[::-1][:min(top_k, len(chunks))]
    return [{"chunk": chunks[int(i)], "score": float(scores[i])} for i in indices]

# Lightweight instruction-tuned generator
GEN_MODEL = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
generator = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL).to(DEVICE)

def generate_text(prompt, max_new_tokens=160):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        output = generator.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

def generate_answer(question):
    return generate_text(
        "Answer the question using your general knowledge. If unsure, say you are unsure.\n"
        f"Question: {question}\nAnswer:"
    )

def rag_query(question, top_k=3):
    hits = retrieve(question, top_k=top_k)
    context = "\n\n".join(
        f"[Chunk {h['chunk']['id']} | {h['chunk']['source']}] {h['chunk']['text']}"
        for h in hits
    )
    prompt = (
        "Answer the question using ONLY the context. If the context does not contain "
        "the answer, say 'The provided documents do not specify.' Do not invent facts.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return {
        "answer": generate_text(prompt),
        "context": context,
        "retrieved_ids": [h["chunk"]["id"] for h in hits],
        "hits": hits
    }

print(f"Loaded {len(documents)} document(s) and created {len(chunks)} chunks.")
print("RAG build complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 878.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 26.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
Device: cpu


Saving sample_rag_test.pdf to sample_rag_test.pdf


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (18 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded 1 document(s) and created 2 chunks.
RAG build complete.


In [ ]:
# CELL 2 — EVALUATE BASELINE vs RAG AND RETRIEVAL

# Create an initial evaluation set from document chunks.
# For rigorous evaluation, replace these auto-generated questions/references
# with human-written questions and verified answers.
eval_data = []
for c in chunks[:min(8, len(chunks))]:
    snippet = c["text"][:500]
    eval_data.append({
        "question": f"What information is provided in this passage from {c['source']}?",
        "reference": snippet,
        "gold_ids": [c["id"]]
    })

def retrieval_scores(question, gold_ids, top_k=3):
    retrieved_ids = [x["chunk"]["id"] for x in retrieve(question, top_k=top_k)]
    gold = set(gold_ids)
    found = set(retrieved_ids) & gold
    precision = len(found) / len(retrieved_ids) if retrieved_ids else 0.0
    recall = len(found) / len(gold) if gold else 0.0
    return precision, recall

rows = []
for i, ex in enumerate(eval_data, 1):
    q = ex["question"]
    retrieved = rag_query(q, top_k=3)
    baseline = generate_answer(q)
    p, r = retrieval_scores(q, ex["gold_ids"], top_k=3)
    rows.append({
        "question": q,
        "reference": ex["reference"],
        "baseline_answer": baseline,
        "rag_answer": retrieved["answer"],
        "retrieval_precision": p,
        "retrieval_recall": r,
        "context": retrieved["context"],
        "retrieved_ids": retrieved["retrieved_ids"]
    })
    print(f"Evaluated {i}/{len(eval_data)}")

results = pd.DataFrame(rows)
display(results[["question", "baseline_answer", "rag_answer",
                 "retrieval_precision", "retrieval_recall"]])
print("Evaluation complete. Answers and retrieved contexts are stored in `results`.")

Evaluated 1/2
Evaluated 2/2


,question,baseline_answer,rag_answer,retrieval_precision,retrieval_recall
0,What information is provided in this passage f...,sample_rag_test.pdf,Classification Predicts categories.,0.5,1.0
1,What information is provided in this passage f...,sample_rag_test.pdf,Classification Predicts categories.,0.5,1.0


Evaluation complete. Answers and retrieved contexts are stored in `results`.


In [ ]:
# CELL 3 — HALLUCINATION / FAITHFULNESS CHECK

from transformers import pipeline

# NLI classifier: predicts entailment, contradiction, or neutral.
nli_device = 0 if torch.cuda.is_available() else -1
nli = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=nli_device
)

def split_claims(answer):
    # Basic sentence splitting; review answers with abbreviations or complex sentences.
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(answer)) if s.strip()]

def nli_label(context, claim):
    output = nli(
        {"text": context, "text_pair": claim},
        truncation=True
    )
    if isinstance(output, list):
        output = output[0]
    return str(output["label"]).lower(), float(output["score"])

def faithfulness_eval(answer, context):
    claims = split_claims(answer)
    if not claims:
        return {"faithfulness": 0.0, "unsupported_rate": 1.0, "details": []}

    details = []
    for claim in claims:
        label, confidence = nli_label(context, claim)
        # Only entailment is counted as supported. Neutral/contradiction are unsupported.
        supported = "entail" in label
        details.append({
            "claim": claim,
            "label": label,
            "confidence": confidence,
            "supported": supported
        })

    faithfulness = sum(d["supported"] for d in details) / len(details)
    return {
        "faithfulness": faithfulness,
        "unsupported_rate": 1.0 - faithfulness,
        "details": details
    }

baseline_rates, rag_rates, baseline_faith, rag_faith, claim_details = [], [], [], [], []
for i, row in results.iterrows():
    base_eval = faithfulness_eval(row["baseline_answer"], row["context"])
    rag_eval = faithfulness_eval(row["rag_answer"], row["context"])
    baseline_rates.append(base_eval["unsupported_rate"])
    rag_rates.append(rag_eval["unsupported_rate"])
    baseline_faith.append(base_eval["faithfulness"])
    rag_faith.append(rag_eval["faithfulness"])
    claim_details.append(rag_eval["details"])
    print(f"Hallucination check {i+1}/{len(results)}")

results["baseline_unsupported_rate"] = baseline_rates
results["rag_unsupported_rate"] = rag_rates
results["baseline_faithfulness"] = baseline_faith
results["rag_faithfulness"] = rag_faith
results["rag_claim_details"] = claim_details

print("Hallucination/faithfulness checks complete.")
print("Unsupported rate is the fraction of answer claims not classified as entailed by the retrieved context.")

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Hallucination check 1/2
Hallucination check 2/2
Hallucination/faithfulness checks complete.
Unsupported rate is the fraction of answer claims not classified as entailed by the retrieved context.


In [ ]:
# CELL 4 — RESULTS, SUMMARY METRICS, AND EXPORT

summary = pd.DataFrame({
    "Metric": [
        "Baseline unsupported-claim rate (hallucination proxy)",
        "RAG unsupported-claim rate (hallucination proxy)",
        "Baseline faithfulness",
        "RAG faithfulness",
        "Retrieval precision@3",
        "Retrieval recall@3"
    ],
    "Value": [
        results["baseline_unsupported_rate"].mean(),
        results["rag_unsupported_rate"].mean(),
        results["baseline_faithfulness"].mean(),
        results["rag_faithfulness"].mean(),
        results["retrieval_precision"].mean(),
        results["retrieval_recall"].mean()
    ]
})

print("OVERALL EVALUATION RESULTS")
display(summary.style.format({"Value": "{:.2%}"}))

print("\nQUESTION-WISE RESULTS")
display(results[[
    "question", "baseline_answer", "rag_answer",
    "baseline_unsupported_rate", "rag_unsupported_rate",
    "baseline_faithfulness", "rag_faithfulness",
    "retrieval_precision", "retrieval_recall"
]])

# Export full details, including contexts and claim-level NLI labels.
output_file = "rag_evaluation_results.csv"
results.to_csv(output_file, index=False)
print(f"Saved detailed results to: {output_file}")
files.download(output_file)

print("\nINTERPRETATION")
print("Hallucination proxy = unsupported claim rate, calculated as 1 - faithfulness.")
print("A lower unsupported-claim rate indicates fewer claims that the NLI model could not verify from retrieved context.")
print("This is an automated proxy, not a perfect ground-truth hallucination measurement; manually review important claims.")

OVERALL EVALUATION RESULTS


,Metric,Value
0,Baseline unsupported-claim rate (hallucination proxy),100.00%
1,RAG unsupported-claim rate (hallucination proxy),100.00%
2,Baseline faithfulness,0.00%
3,RAG faithfulness,0.00%
4,Retrieval precision@3,50.00%
5,Retrieval recall@3,100.00%



QUESTION-WISE RESULTS


,question,baseline_answer,rag_answer,baseline_unsupported_rate,rag_unsupported_rate,baseline_faithfulness,rag_faithfulness,retrieval_precision,retrieval_recall
0,What information is provided in this passage f...,sample_rag_test.pdf,Classification Predicts categories.,1.0,1.0,0.0,0.0,0.5,1.0
1,What information is provided in this passage f...,sample_rag_test.pdf,Classification Predicts categories.,1.0,1.0,0.0,0.0,0.5,1.0


Saved detailed results to: rag_evaluation_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


INTERPRETATION
Hallucination proxy = unsupported claim rate, calculated as 1 - faithfulness.
A lower unsupported-claim rate indicates fewer claims that the NLI model could not verify from retrieved context.
This is an automated proxy, not a perfect ground-truth hallucination measurement; manually review important claims.
